# Unconstrained Flux Sampling — CoDA vs Euclidean WSSR Comparison

Uses **1000 ACHR-sampled flux states** from `flux_range.ipynb` where only R1 = 1 is fixed.  
All other fluxes (including R16–R20 effluxes) vary freely across the feasible polytope.

For each sample the **true** v2, v8, v9 are known. We then simulate noisy MID observations  
and ask: which objective recovers the true fluxes most accurately?

| Method | Transform | Space |
|--------|-----------|-------|
| **Euclidean** | none | simplex (rank-deficient) |
| **CLR** | log xᵢ − mean(log x) | ℝⁿ (rank n−1) |
| **ALR** | log(xᵢ / x₀) | ℝⁿ⁻¹ |
| **ILR** | Gram-Schmidt log-contrast | ℝⁿ⁻¹ (isometric) |
| **LeaveOneOut** | drop last + scale | ℝⁿ⁻¹ |

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import optimistix
import matplotlib.pyplot as plt
import seaborn as sns

from jax import config as jax_config
jax_config.update("jax_enable_x64", True)

from tca_openflux import (
    TCA_OpenFLux, objective_wssr, objective_clr_wssr,
    objective_alr_wssr, objective_ilr_wssr,
    objective_leave_one_out_wssr,
)
from emu_example import closure, clr_func, ilr_func
from helper.coda import alr_func

METHODS  = ["Euclidean", "CLR", "ALR", "ILR", "LeaveOneOut"]
MARKERS  = {"Euclidean": "o", "CLR": "s", "ALR": "D", "ILR": "v", "LeaveOneOut": "^"}
PALETTE  = sns.color_palette("colorblind", n_colors=5)

## Core estimation function

Unlike `alr_ilr_comparison.ipynb`, the efflux values (v16–v20) are **not hardcoded** —  
they come from the sampled flux state so the model sees the correct label dynamics.

In [ ]:
def fva_full(parameters: dict, fixed: dict) -> dict:
    """Run all 5 methods for one flux state.

    Parameters
    ----------
    parameters : dict  free fluxes to estimate  {v2, v8, v9}
    fixed      : dict  per-sample fixed fluxes   {v1, v16, v17, v18, v19, v20}
    """
    # ── True MIDs ──────────────────────────────────────────────────────────────
    true_data = TCA_OpenFLux(parameters=parameters, fixed=fixed)

    # ── Simulate noisy observations (1000 draws, CV = 10 %) ───────────────────
    key = jax.random.PRNGKey(1234)
    sims = 1000
    total_isos  = jax.tree.map(lambda x: x * 1e5, true_data)
    norm_sigma  = jax.tree.map(lambda x: x * 0.1, total_isos)
    _, key_sim  = jax.random.split(key)
    norm_sim    = jax.tree.map(
        lambda m, s: m + s * jax.random.normal(key_sim, (sims, m.shape[0])),
        total_isos, norm_sigma,
    )
    iso_prop = jax.tree.map(closure, norm_sim)

    # ── Euclidean stats ────────────────────────────────────────────────────────
    mean_sim = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_prop)
    std_sim  = jax.tree.map(lambda x: jnp.std( x, axis=0), iso_prop)
    D_sim    = jax.tree.map(jnp.diag, std_sim)

    # ── CLR stats ──────────────────────────────────────────────────────────────
    iso_clr  = jax.tree.map(lambda m: jax.vmap(clr_func)(m), iso_prop)
    mean_clr = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_clr)
    std_clr  = jax.tree.map(lambda x: jnp.std( x, axis=0), iso_clr)
    D_clr    = jax.tree.map(jnp.diag, std_clr)

    # ── ALR stats ──────────────────────────────────────────────────────────────
    iso_alr  = jax.tree.map(lambda m: jax.vmap(alr_func)(m), iso_prop)
    mean_alr = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_alr)
    std_alr  = jax.tree.map(lambda x: jnp.std( x, axis=0), iso_alr)
    D_alr    = jax.tree.map(jnp.diag, std_alr)

    # ── ILR stats ──────────────────────────────────────────────────────────────
    iso_ilr  = jax.tree.map(lambda m: jax.vmap(ilr_func)(m), iso_prop)
    mean_ilr = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_ilr)
    std_ilr  = jax.tree.map(lambda x: jnp.std( x, axis=0), iso_ilr)
    D_ilr    = jax.tree.map(jnp.diag, std_ilr)

    # ── Init from true fluxes (no fixed guess bias since samples are unconstrained) ──
    init_params = {k: float(v) for k, v in parameters.items()}

    # ── Optimise (throw=False: return best-found result instead of crashing) ───
    solver = optimistix.BFGS(rtol=1e-12, atol=1e-12, verbose=frozenset())

    result_euc = optimistix.minimise(
        objective_wssr, y0=init_params,
        args=(mean_sim, D_sim, fixed), solver=solver, max_steps=10000, throw=False,
    )
    result_clr = optimistix.minimise(
        objective_clr_wssr, y0=init_params,
        args=(mean_clr, D_clr, fixed), solver=solver, max_steps=10000, throw=False,
    )
    result_alr = optimistix.minimise(
        objective_alr_wssr, y0=init_params,
        args=(mean_alr, D_alr, fixed), solver=solver, max_steps=10000, throw=False,
    )
    result_ilr = optimistix.minimise(
        objective_ilr_wssr, y0=init_params,
        args=(mean_ilr, D_ilr, fixed), solver=solver, max_steps=10000, throw=False,
    )

    init_params_loo = {
        "fluxes": init_params,
        "scales": jax.tree.map(lambda _: 1.0, mean_sim),
    }
    result_loo = optimistix.minimise(
        objective_leave_one_out_wssr, y0=init_params_loo,
        args=(mean_sim, D_sim, fixed), solver=solver, max_steps=10000, throw=False,
    )

    true_mids = {k: np.array(v) for k, v in true_data.items()}

    return {
        "Euclidean":   result_euc.value,
        "CLR":         result_clr.value,
        "ALR":         result_alr.value,
        "ILR":         result_ilr.value,
        "LeaveOneOut": result_loo.value["fluxes"],
        "true_mids":   true_mids,
    }

## Load unconstrained flux samples

Generated by the new cell in `flux_range.ipynb` — 1000 ACHR polytope samples  
with only R1 = 1 fixed. All columns are normalised to the v1 = 1 scale.

In [ ]:
flux_df_raw = pd.read_csv("flux_sampling/flux_samples_unconstrained_1000.csv", index_col=0)
print(f"Loaded {len(flux_df_raw)} unconstrained samples.")

# Pre-filter: remove samples where any efflux (R16-R20) or free flux (R2, R8, R9)
# is below a safe minimum. Near-zero values cause log(~0) → -inf in CLR/ALR,
# and near-zero fluxes in the EMU system make the linear solve ill-conditioned.
MIN_FLUX = 1e-3
efflux_cols = ["R16", "R17", "R18", "R19", "R20"]
free_cols   = ["R2", "R8", "R9"]
keep = (
    (flux_df_raw[efflux_cols] >= MIN_FLUX).all(axis=1) &
    (flux_df_raw[free_cols]   >= MIN_FLUX).all(axis=1)
)
flux_df = flux_df_raw[keep].reset_index(drop=True)
print(f"After filtering (all efflux + free flux >= {MIN_FLUX}): {len(flux_df)} samples retained "
      f"({len(flux_df_raw) - len(flux_df)} removed).")

cols_of_interest = efflux_cols + free_cols
print("\nSummary of key fluxes (normalised, v1 = 1):")
print(flux_df[cols_of_interest].describe().round(4))

## Batch run — all 1000 samples

For each sample the efflux values (R16–R20) are passed as `fixed` so the EMU model  
uses the correct flux state; the optimizer then estimates v2, v8, v9 blind to the truth.

In [ ]:
flux_vars = ["v2", "v8", "v9"]

batch_true   = []
batch_fixed  = []
batch_results = {m: [] for m in METHODS}
batch_true_mids = []

for i, (_, row) in enumerate(flux_df.iterrows()):
    params = {"v2": float(row["R2"]), "v8": float(row["R8"]), "v9": float(row["R9"])}
    fixed  = {
        "v1":  1.0,
        "v16": float(row["R16"]),
        "v17": float(row["R17"]),
        "v18": float(row["R18"]),
        "v19": float(row["R19"]),
        "v20": float(row["R20"]),
    }
    result = fva_full(params, fixed)
    batch_true.append(params)
    batch_fixed.append(fixed)
    for m in METHODS:
        batch_results[m].append(result[m])
    batch_true_mids.append(result["true_mids"])

    if (i + 1) % 100 == 0:
        print(f"  {i + 1} / {len(flux_df)} done")

print(f"\nFinished {len(batch_true)} runs.")

## Build tidy DataFrame + IQR outlier detection

In [ ]:
true_arr = {v: np.array([d[v] for d in batch_true]) for v in flux_vars}

records = []
for i in range(len(batch_true)):
    for v in flux_vars:
        for method in METHODS:
            records.append({
                "Sample":    i,
                "Flux":      v,
                "Method":    method,
                "True":      true_arr[v][i],
                "Estimated": float(batch_results[method][i][v]),
                "Residual":  float(batch_results[method][i][v]) - true_arr[v][i],
            })

df = pd.DataFrame(records)

# IQR outlier detection on the worst absolute residual per sample
abs_res = df.groupby("Sample")["Residual"].apply(lambda x: np.abs(x).max())
q1, q3  = abs_res.quantile(0.25), abs_res.quantile(0.75)
iqr     = q3 - q1
outlier_samples = set(abs_res[abs_res > q3 + 1.5 * iqr].index.tolist())

df["Outlier"] = df["Sample"].isin(outlier_samples)
df_clean = df[~df["Outlier"]]
n_out = len(outlier_samples)
print(f"Outliers removed: {n_out}/{len(batch_true)}")
print(f"Clean samples:    {len(batch_true) - n_out}")

## Residual plots (5 methods)

In [ ]:
sns.set_theme(style="whitegrid", palette="colorblind")
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    f"Residuals — {len(batch_true) - n_out} clean samples "
    f"({n_out} outlier{'s' if n_out != 1 else ''} removed by IQR) "
    f"| unconstrained 1000-sample run",
    fontsize=12, y=1.01,
)

for col, v in enumerate(flux_vars):
    sub = df_clean[df_clean["Flux"] == v]

    # Row 0 — parity
    ax = axes[0, col]
    lo = min(sub["True"].min(), sub["Estimated"].min())
    hi = max(sub["True"].max(), sub["Estimated"].max())
    pad = (hi - lo) * 0.05
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1)
    sns.scatterplot(
        data=sub, x="True", y="Estimated",
        hue="Method", style="Method", markers=MARKERS,
        hue_order=METHODS, style_order=METHODS,
        palette="colorblind", s=25, alpha=0.6,
        ax=ax, legend=(col == 0),
    )
    ax.set_title(f"{v} — parity")
    ax.set_xlabel(f"True {v}")
    ax.set_ylabel(f"Estimated {v}")
    if col == 0:
        ax.legend(fontsize=7, title="Method")

    # Row 1 — residuals
    ax = axes[1, col]
    ax.axhline(0, color="black", lw=1, ls="--", zorder=0)
    sns.scatterplot(
        data=sub, x="True", y="Residual",
        hue="Method", style="Method", markers=MARKERS,
        hue_order=METHODS, style_order=METHODS,
        palette="colorblind", s=25, alpha=0.6,
        ax=ax, legend=False,
    )
    for method, colour in zip(METHODS, PALETTE):
        bias = sub.loc[sub["Method"] == method, "Residual"].mean()
        ax.axhline(bias, color=colour, lw=1.2, ls=":", alpha=0.9)
    ax.set_title(f"{v} — residuals")
    ax.set_xlabel(f"True {v}")
    ax.set_ylabel("Residual (estimated − true)")

fig.tight_layout()
plt.savefig("unconstrained_parity_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary statistics (Bias / MAE / RMSE)

In [ ]:
rows = []
for v in flux_vars:
    for method in METHODS:
        err = df_clean.loc[
            (df_clean["Flux"] == v) & (df_clean["Method"] == method),
            "Residual",
        ].values
        rows.append({
            "Flux": v, "Method": method,
            "Bias": float(err.mean()),
            "MAE":  float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
        })

summary = pd.DataFrame(rows).set_index(["Flux", "Method"])
pd.set_option("display.float_format", "{:.5f}".format)
print(f"Summary statistics ({len(batch_true) - n_out} clean samples):")
print(summary)

# Heatmap of RMSE for quick visual comparison
rmse_pivot = summary["RMSE"].unstack("Method")[METHODS]
fig, ax = plt.subplots(figsize=(9, 3))
sns.heatmap(
    rmse_pivot, annot=True, fmt=".5f", cmap="YlOrRd",
    linewidths=0.5, ax=ax,
)
ax.set_title("RMSE by flux and method (lower = better)")
fig.tight_layout()
plt.savefig("unconstrained_rmse_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Recovery rate

In [ ]:
tolerances = [0.01, 0.05, 0.10]
recovery_rows = []
for v in flux_vars:
    for method in METHODS:
        sub = df_clean[(df_clean["Flux"] == v) & (df_clean["Method"] == method)]
        rel_err = (sub["Residual"] / sub["True"]).abs()
        for tol in tolerances:
            recovery_rows.append({
                "Flux": v, "Method": method,
                "Tolerance": f"\u00b1{int(tol*100)}%",
                "Recovery (%)": round(100 * (rel_err <= tol).mean(), 1),
            })

recovery_df = pd.DataFrame(recovery_rows)
print(recovery_df.pivot_table(
    index=["Flux", "Method"], columns="Tolerance", values="Recovery (%)"
).to_string())

g = sns.FacetGrid(recovery_df, col="Tolerance", height=4, aspect=0.9, sharey=True)
g.map_dataframe(
    sns.barplot, x="Flux", y="Recovery (%)", hue="Method",
    hue_order=METHODS, order=flux_vars,
    palette="colorblind", errorbar=None,
)
g.add_legend()
g.set_axis_labels("Flux", "Recovery (%)")
g.set_titles(col_template="{col_name}")
g.figure.suptitle(
    "Recovery rate by tolerance — 5 methods | unconstrained 1000-sample run", y=1.03
)
for ax in g.axes.flat:
    ax.set_ylim(0, 105)
    ax.axhline(100, color="grey", lw=0.8, ls="--")
g.tight_layout()
plt.savefig("unconstrained_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

## Error correlation heatmap

Shows how estimation errors in v2, v8, v9 co-vary across methods.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle("Residual correlation across fluxes (clean samples)", fontsize=11)

for ax, method, colour in zip(axes, METHODS, PALETTE):
    sub = df_clean[df_clean["Method"] == method].pivot(
        index="Sample", columns="Flux", values="Residual"
    )[flux_vars]
    corr = sub.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr, ax=ax, annot=True, fmt=".2f", vmin=-1, vmax=1,
        cmap="coolwarm", square=True, linewidths=0.5,
        mask=mask, cbar=(ax == axes[-1]),
    )
    ax.set_title(method)
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.tight_layout()
plt.savefig("unconstrained_error_corr.png", dpi=150, bbox_inches="tight")
plt.show()

## Effect of efflux magnitude on estimation error

Because R16–R20 vary freely here (unlike the constrained run), we can test whether  
the efflux magnitude influences how well each method recovers v2, v8, v9.

In [ ]:
# Attach per-sample efflux magnitudes to the tidy df
efflux_cols = ["R16", "R17", "R18", "R19", "R20"]
efflux_sum  = flux_df[efflux_cols].sum(axis=1).values
sample_meta = pd.DataFrame({
    "Sample":       range(len(batch_true)),
    "efflux_total": efflux_sum,
    "v8_true":      true_arr["v8"],
})
df_clean = df_clean.merge(sample_meta, on="Sample", how="left")

# Bin total efflux into quartiles
df_clean["efflux_bin"] = pd.qcut(
    df_clean["efflux_sum"] if "efflux_sum" in df_clean.columns else df_clean["efflux_total"],
    q=4, labels=["Q1", "Q2", "Q3", "Q4"],
)

# RMSE per method per efflux quartile for v8 (most sensitive flux)
rmse_by_efflux = (
    df_clean[df_clean["Flux"] == "v8"]
    .groupby(["efflux_bin", "Method"])["Residual"]
    .apply(lambda x: np.sqrt((x**2).mean()))
    .reset_index(name="RMSE")
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(
    data=rmse_by_efflux, x="efflux_bin", y="RMSE",
    hue="Method", hue_order=METHODS,
    palette="colorblind", ax=ax,
)
ax.set_title("v8 RMSE vs total efflux quartile (Q1 = low, Q4 = high)")
ax.set_xlabel("Total efflux (R16+…+R20) quartile")
ax.set_ylabel("RMSE")
ax.legend(fontsize=8)
fig.tight_layout()
plt.savefig("unconstrained_efflux_rmse.png", dpi=150, bbox_inches="tight")
plt.show()

## True MID ranges across all unconstrained samples

Broader flux variation → broader MID spread. This visualises how much harder  
the estimation problem is compared to the constrained (R16–R20 fixed) run.

In [ ]:
mid_records = []
mid_keys = list(batch_true_mids[0].keys())

for i, mids in enumerate(batch_true_mids):
    for key, values in mids.items():
        for iso_idx, val in enumerate(values):
            mid_records.append({
                "Sample":       i,
                "Metabolite":   key,
                "Isotopologue": f"m+{iso_idx}",
                "iso_idx":      iso_idx,
                "MID":          float(val),
            })

mid_df = pd.DataFrame(mid_records)

met_labels = {
    "lysx_123456": "LysX (123456)",
    "valx_12345":  "ValX (12345)",
    "aspx_1234":   "OAA (1234)",
    "suc_1234":    "SUC (1234)",
}

sns.set_theme(style="whitegrid", palette="colorblind")
fig, axes = plt.subplots(1, len(mid_keys), figsize=(4 * len(mid_keys), 4), sharey=True)
fig.suptitle(
    "True MID ranges across 1000 unconstrained samples", fontsize=12, y=1.02
)

for ax, key in zip(axes, mid_keys):
    sub = mid_df[mid_df["Metabolite"] == key].sort_values("iso_idx")

    for _, grp in sub.groupby("Sample"):
        ax.plot(grp["Isotopologue"], grp["MID"],
                color="steelblue", alpha=0.05, lw=0.8, zorder=1)

    stats_df = (
        sub.groupby(["iso_idx", "Isotopologue"])["MID"]
        .agg(mean="mean", std="std")
        .reset_index()
        .sort_values("iso_idx")
    )
    ax.fill_between(
        stats_df["Isotopologue"],
        (stats_df["mean"] - stats_df["std"]).clip(lower=0),
        (stats_df["mean"] + stats_df["std"]).clip(upper=1),
        alpha=0.25, color="steelblue",
    )
    ax.plot(stats_df["Isotopologue"], stats_df["mean"],
            color="steelblue", lw=2, marker="o", ms=5, zorder=3)

    ax.set_title(met_labels.get(key, key), fontsize=10)
    ax.set_xlabel("Isotopologue")
    ax.tick_params(axis="x", rotation=45)

axes[0].set_ylabel("MID (fraction)")
fig.tight_layout()
plt.savefig("unconstrained_true_mids.png", dpi=150, bbox_inches="tight")
plt.show()